# 04 - Embedding Generation

Generate vector embeddings for Forge knowledge chunks using an embedding model. These embeddings will later be stored in a vector database for semantic retrieval.

In [10]:
!pip install -q sentence-transformers

In [1]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd


from google.colab import drive

from sentence_transformers import SentenceTransformer

In [2]:
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
PROJECT_ROOT = Path("/content/drive/MyDrive/forge")

CONFIG = {
    "project_root": PROJECT_ROOT,
    "chunks_file": PROJECT_ROOT / "knowledge_base" / "chunks.json",
    "embeddings": PROJECT_ROOT / "knowledge_base" / "embeddings",
}

print(CONFIG["chunks_file"])

/content/drive/MyDrive/forge/knowledge_base/chunks.json


In [4]:
print(CONFIG["chunks_file"].exists())

True


In [5]:
with open(CONFIG["chunks_file"], "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(len(chunks))

3944


In [6]:
MODEL_NAME = "BAAI/bge-base-en-v1.5"

model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [7]:
texts = [chunk["text"] for chunk in chunks]

print(f"Total texts: {len(texts)}")
print(f"Sample text length: {len(texts[0])}")

Total texts: 3944
Sample text length: 928


In [9]:
start_time = time.time()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

end_time = time.time()

print(f"Embedding generation completed.")
print(f"Time taken: {(end_time - start_time)/60:.2f} minutes")
print(f"Embedding shape: {embeddings.shape}")

Batches:   0%|          | 0/124 [00:00<?, ?it/s]

Embedding generation completed.
Time taken: 1.40 minutes
Embedding shape: (3944, 768)


In [10]:
embedding_file = CONFIG["project_root"] / "knowledge_base" / "embeddings.npy"

np.save(
    embedding_file,
    embeddings
)

print(f"Saved embeddings: {embedding_file}")
print(f"Shape: {embeddings.shape}")

Saved embeddings: /content/drive/MyDrive/forge/knowledge_base/embeddings.npy
Shape: (3944, 768)


In [11]:
metadata = []

for chunk in chunks:
    metadata.append({
        "chunk_id": chunk["chunk_id"],
        "technology": chunk["technology"],
        "source": chunk["source"],
        "path": chunk["path"],
        "chunk_index": chunk["chunk_index"],
        "text": chunk["text"],
        "character_count": chunk["character_count"]
    })

metadata_file = CONFIG["project_root"] / "knowledge_base" / "embedding_metadata.json"

with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False)

print(f"Saved metadata: {metadata_file}")
print(f"Metadata entries: {len(metadata)}")

Saved metadata: /content/drive/MyDrive/forge/knowledge_base/embedding_metadata.json
Metadata entries: 3944


In [12]:
loaded_embeddings = np.load(
    CONFIG["project_root"] / "knowledge_base" / "embeddings.npy"
)

with open(
    CONFIG["project_root"] / "knowledge_base" / "embedding_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    loaded_metadata = json.load(f)

print("Embedding shape:", loaded_embeddings.shape)
print("Metadata count :", len(loaded_metadata))

print("\nFirst metadata entry:")
print(loaded_metadata[0])

Embedding shape: (3944, 768)
Metadata count : 3944

First metadata entry:
{'chunk_id': '0e84d87b-c73d-412a-8dcb-9b9aaac03f98', 'technology': 'anthropic_claude', 'source': 'api_reference', 'path': 'anthropic_claude/api_reference.txt', 'chunk_index': 0, 'text': "The Claude API is a RESTful API at https://api.anthropic.com that provides programmatic access to Claude models and Claude Managed Agents.\nNew to Claude? For direct model access, start with Get started and Working with Messages. For managed agent infrastructure, see the Claude Managed Agents quickstart.\nTo use the Claude API, you'll need:\nFor step-by-step setup instructions, see Get started.\nThe Claude API includes the following APIs:\nGeneral Availability:\nPOST /v1/messages)POST /v1/messages/batches)POST /v1/messages/count_tokens)GET /v1/models)Beta:\nPOST /v1/files, GET /v1/files)POST /v1/skills, GET /v1/skills)POST /v1/agents, GET /v1/agents)POST /v1/sessions, GET /v1/sessions/{id}/stream)POST /v1/environments, GET /v1/en